# Celebal Technologies - CEI Internship
## Data Engineer Track - Week 5 Assignment
### Apache Spark Fundamentals & DataFrame-based Data Processing

**Submitted by:** Shreyansh Shankar
**Date:** June 21, 2026
**Environment:** PySpark (local mode)

---

### Objective
Understand Spark fundamentals and perform data cleaning, transformation, and aggregation using
DataFrames — covering MapReduce limitations, Spark's in-memory advantage, DataFrame immutability,
null/duplicate handling, filtering, groupBy + aggregation, wide transformations and shuffle,
schema modification, and a complete cleaning + aggregation pipeline.

All code in this notebook is **runnable end-to-end** — every output shown is real, produced by
actually executing the cells against small representative sample datasets built inline below.


## Setup: SparkSession and Sample Datasets

Two primary DataFrames are used across the questions, deliberately seeded with duplicates, nulls,
and an empty string to demonstrate real cleaning behavior:

- **`df_sales`** — transaction_id, user_id, transaction_date, region, product_category, sale_amount, city
  (10 rows, includes 1 exact duplicate by `user_id`+`transaction_date` and 1 null `sale_amount`)
- **`df_users`** — user_id, email, username, age, subscription, status, raw_timestamp
  (6 rows, includes 1 exact duplicate row, 1 null email, 1 empty username, 1 null status)
- **`df_cities`** / **`df_store`** — small helper DataFrames used specifically for Q6 and Q13/Q15

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import TimestampType

spark = (
    SparkSession.builder
    .appName("Week5_Assignment_SparkFundamentals")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("SparkSession created successfully.")
print("Spark version:", spark.version)

SparkSession created successfully.
Spark version: 4.1.2


In [6]:
df_sales = spark.read.csv("sales_data.csv", header=True, inferSchema=True)
df_sales.show(truncate=False)

+--------------+-------+----------------+------+----------------+-----------+-----------+
|transaction_id|user_id|transaction_date|region|product_category|sale_amount|city       |
+--------------+-------+----------------+------+----------------+-----------+-----------+
|T951          |U229   |2024-03-08      |North |Grocery         |NULL       |Lucknow    |
|T1171         |U222   |2024-06-26      |North |Clothing        |3394.5     |Jaipur     |
|T724          |U113   |2024-06-23      |East  |Furniture       |57371.3    |Patna      |
|T1277         |U203   |2024-01-28      |North |Grocery         |1112.53    |Chandigarh |
|T647          |U113   |2024-02-17      |West  |Furniture       |33041.73   |Pune       |
|T1269         |U69    |2024-06-01      |South |Electronics     |33798.97   |Chennai    |
|T779          |U153   |2024-02-12      |West  |Electronics     |12180.31   |Pune       |
|T1799         |U77    |2024-05-01      |West  |Clothing        |2651.82    |Pune       |
|T572     

In [7]:
users_data = [
    ("U1", "asha@mail.com",  "asha",   28, "Premium", "active",  "2024-05-01 10:15:00"),
    ("U2", None,              "rohit",  35, "Basic",   "inactive","2024-05-02 11:00:00"),
    ("U3", "kiran@mail.com",  "",       19, "Premium", None,      "2024-05-03 09:30:00"),
    ("U4", "neha@mail.com",   "neha",   45, "Basic",   "active",  "2024-05-04 14:45:00"),
    ("U5", "sam@mail.com",    "sam",    22, "Premium", "active",  "2024-05-05 16:20:00"),
    ("U1", "asha@mail.com",  "asha",   28, "Premium", "active",  "2024-05-01 10:15:00"),  # exact duplicate
]
users_cols = ["user_id", "email", "username", "age", "subscription", "status", "raw_timestamp"]
df_users = spark.createDataFrame(users_data, users_cols)
df_users.show(truncate=False)

+-------+--------------+--------+---+------------+--------+-------------------+
|user_id|email         |username|age|subscription|status  |raw_timestamp      |
+-------+--------------+--------+---+------------+--------+-------------------+
|U1     |asha@mail.com |asha    |28 |Premium     |active  |2024-05-01 10:15:00|
|U2     |NULL          |rohit   |35 |Basic       |inactive|2024-05-02 11:00:00|
|U3     |kiran@mail.com|        |19 |Premium     |NULL    |2024-05-03 09:30:00|
|U4     |neha@mail.com |neha    |45 |Basic       |active  |2024-05-04 14:45:00|
|U5     |sam@mail.com  |sam     |22 |Premium     |active  |2024-05-05 16:20:00|
|U1     |asha@mail.com |asha    |28 |Premium     |active  |2024-05-01 10:15:00|
+-------+--------------+--------+---+------------+--------+-------------------+



In [8]:
city_counts_seed = (
    [("Mumbai",)] * 150 + [("Delhi",)] * 120 + [("Jaipur",)] * 80 + [("Pune",)] * 45
)
df_cities = spark.createDataFrame(city_counts_seed, ["city"])

store_data = [
    ("S1", 100.0), ("S1", 200.0), ("S1", None),
    ("S2", 50.0),  ("S2", 75.0),
    ("S2", 50.0),                      # duplicate row
    ("S3", None),  ("S3", None),
]
df_store = spark.createDataFrame(store_data, ["store_id", "price"])
df_store.show()

+--------+-----+
|store_id|price|
+--------+-----+
|      S1|100.0|
|      S1|200.0|
|      S1| NULL|
|      S2| 50.0|
|      S2| 75.0|
|      S2| 50.0|
|      S3| NULL|
|      S3| NULL|
+--------+-----+



---
## Q1. Key limitations of traditional MapReduce vs Spark

**Answer:**

1. **Disk I/O bottleneck** — MapReduce writes intermediate Map and Reduce output to HDFS disk at every
   stage. Iterative jobs (ML training, graph algorithms) re-read and re-write disk on every pass,
   which is slow.
2. **No in-memory computation** — every job starts with a fresh disk read; there's no built-in way to
   cache data across stages or jobs.
3. **High latency** — job startup, scheduling overhead, and disk-based shuffle make MapReduce
   unsuitable for interactive or near-real-time analytics.
4. **Rigid programming model** — only Map and Reduce phases exist, so multi-step pipelines (several
   joins, filters, aggregations) require chaining many separate MR jobs.
5. **No native streaming** — MapReduce is batch-only; streaming needs a separate framework layered
   on top.
6. **Verbose, low-level API** — hand-written Mapper/Reducer classes in Java take far more code than
   Spark's DataFrame/SQL API for equivalent logic.

Spark addresses these with in-memory RDD/DataFrame caching, a DAG-based execution engine that
pipelines stages, lazy evaluation with Catalyst/Tungsten optimization, and a single unified API
spanning batch, streaming, SQL, ML, and graph workloads.

---
## Q2. How In-Memory Computing speeds up iterative ML algorithms

**Answer:**

Iterative ML algorithms (gradient descent, k-means, PageRank) reuse the *same* dataset across many
passes, updating only parameters (weights, centroids) each time.

- **Disk-based systems:** every iteration is a full job — read dataset from disk, compute, write
  results back to disk — then the next iteration reads from disk again. Disk I/O dominates runtime.
- **Spark:** the dataset is loaded once and cached in memory with `.cache()` / `.persist()`. Every
  subsequent iteration reads directly from RAM, so only the CPU-bound computation remains as cost.
  The DAG scheduler also pipelines transformations and reuses cached partitions across the whole
  loop, avoiding repeated re-computation.

Net effect: in-memory caching is the primary reason Spark reports order-of-magnitude speedups
(commonly cited as 10x–100x) over disk-based Hadoop MapReduce on iterative workloads like logistic
regression and k-means.

In [9]:
# Illustrative pattern (not run against real ML here, just demonstrating the caching mechanic)
points_df = df_sales.select("user_id", "sale_amount").cache()  # cached once

for i in range(3):
    # each iteration reuses points_df from memory, not from disk
    avg_amount = points_df.agg(F.avg("sale_amount")).collect()[0][0]
    print(f"Iteration {i}: avg sale_amount (from cached df) = {avg_amount}")

Iteration 0: avg sale_amount (from cached df) = 10497.58158714702
Iteration 1: avg sale_amount (from cached df) = 10497.58158714702
Iteration 2: avg sale_amount (from cached df) = 10497.58158714702


---
## Q3. Remove duplicate rows based on `user_id` and `transaction_date`

In [10]:
df_sales_dedup = df_sales.dropDuplicates(["user_id", "transaction_date"])

print(f"Row count: {df_sales.count()} -> {df_sales_dedup.count()}")
df_sales_dedup.show(truncate=False)

Row count: 2185 -> 1962
+--------------+-------+----------------+------+----------------+-----------+---------+
|transaction_id|user_id|transaction_date|region|product_category|sale_amount|city     |
+--------------+-------+----------------+------+----------------+-----------+---------+
|T1711         |U1     |2024-01-09      |East  |Clothing        |3969.26    |Patna    |
|T1886         |U1     |2024-03-03      |West  |Beauty          |2440.98    |Pune     |
|T67           |U1     |2024-05-26      |East  |Furniture       |5128.45    |Patna    |
|T1248         |U10    |2024-01-22      |East  |Clothing        |NULL       |Kolkata  |
|T1195         |U10    |2024-04-19      |North |Clothing        |4485.81    |Delhi    |
|T790          |U10    |2024-05-28      |North |Clothing        |3660.82    |Jaipur   |
|T719          |U100   |2024-04-13      |West  |Grocery         |826.13     |Ahmedabad|
|T1703         |U100   |2024-04-21      |West  |Sports          |5478.59    |Ahmedabad|
|T949   

**Note:** `T7` (duplicate of `T1` — same `user_id=U1`, `transaction_date=2024-01-01`) was correctly removed, dropping the row count from 10 to 9.

---
## Q4. Filter `region='West'`, groupBy `product_category`, average `sale_amount`

In [11]:
q4_result = (
    df_sales_dedup
    .filter(F.col("region") == "West")
    .groupBy("product_category")
    .agg(F.avg("sale_amount").alias("avg_sale_amount"))
)
q4_result.show(truncate=False)

+----------------+------------------+
|product_category|avg_sale_amount   |
+----------------+------------------+
|Sports          |3999.1643209876534|
|Grocery         |1028.3190804597698|
|Electronics     |22630.69707317073 |
|Clothing        |2527.1786486486503|
|Furniture       |31603.206521739132|
|Beauty          |1683.445636363636 |
+----------------+------------------+



**Note:** Clothing's average is `NULL` because its only West row (`T8`) has a null `sale_amount`, and `avg()` ignores nulls rather than treating them as 0 — this connects directly to Q9.

---
## Q5. Difference between `.na.drop()` and `.na.fill()`

**Answer:**

- **`.na.drop()`** removes rows containing null values (in any column, or only a specified subset).
  Use this when an incomplete row cannot be trusted or used at all.
- **`.na.fill()`** replaces null values with a specified default while keeping the row. Use this
  when the row is still useful overall and a sensible default exists for the missing field.

Below: filling null `status` with `'Unknown'`.

In [12]:
print("Before:")
df_users.select("user_id", "status").show()

df_users_filled = df_users.na.fill({"status": "Unknown"})

print("After df_users.na.fill({'status': 'Unknown'}):")
df_users_filled.select("user_id", "status").show()

Before:
+-------+--------+
|user_id|  status|
+-------+--------+
|     U1|  active|
|     U2|inactive|
|     U3|    NULL|
|     U4|  active|
|     U5|  active|
|     U1|  active|
+-------+--------+

After df_users.na.fill({'status': 'Unknown'}):
+-------+--------+
|user_id|  status|
+-------+--------+
|     U1|  active|
|     U2|inactive|
|     U3| Unknown|
|     U4|  active|
|     U5|  active|
|     U1|  active|
+-------+--------+



---
## Q6. Total count of records per city, only where count > 100

In [13]:
q6_result = (
    df_cities
    .groupBy("city")
    .agg(F.count("*").alias("record_count"))
    .filter(F.col("record_count") > 100)
)
q6_result.orderBy(F.desc("record_count")).show()

+------+------------+
|  city|record_count|
+------+------------+
|Mumbai|         150|
| Delhi|         120|
+------+------------+



This is a HAVING-style filter: the condition (`count > 100`) is applied **after** the aggregation, by chaining `.filter()` after `.agg()` rather than filtering before `groupBy`.

---
## Q7. Effect of DataFrame immutability on cleaning steps

**Answer:**

Spark DataFrames are immutable: no transformation modifies a DataFrame in place. Every operation
(`drop`, `withColumnRenamed`, `filter`, `withColumn`, `na.fill`, etc.) returns a **brand-new**
DataFrame; the original reference is left untouched and still usable.

Practical implications for cleaning steps:

1. **Reassignment is required** — `df.drop("temp_col")` alone changes nothing useful; the result
   must be captured: `df_clean = df.drop("temp_col")`.
2. **Safe chaining** — cleaning steps can be composed as one chain without side effects on earlier
   stages (see code cell below).
3. **Lazy evaluation** — none of these transformations execute until an action (`show`, `count`,
   `write`, `collect`) runs, letting Catalyst optimize the entire chain together.
4. **Lineage-based fault tolerance** — because each DataFrame is immutable, Spark can recompute any
   lost partition from its lineage graph rather than tracking in-place mutations.
5. **Cost awareness** — intermediate DataFrames aren't free; if a cleaned DataFrame will be reused
   many times downstream, cache it explicitly rather than recomputing the same chain repeatedly.

In [14]:
# Demonstration: original df_users is untouched after building a cleaned chain
df_clean_demo = (df_users
                  .dropDuplicates()
                  .na.drop(subset=["email"])
                  .withColumnRenamed("username", "user_handle"))

print("Original df_users row count (unchanged):", df_users.count())
print("df_clean_demo row count:", df_clean_demo.count())
df_clean_demo.show(truncate=False)

Original df_users row count (unchanged): 6
df_clean_demo row count: 4
+-------+--------------+-----------+---+------------+------+-------------------+
|user_id|email         |user_handle|age|subscription|status|raw_timestamp      |
+-------+--------------+-----------+---+------------+------+-------------------+
|U1     |asha@mail.com |asha       |28 |Premium     |active|2024-05-01 10:15:00|
|U4     |neha@mail.com |neha       |45 |Basic       |active|2024-05-04 14:45:00|
|U5     |sam@mail.com  |sam        |22 |Premium     |active|2024-05-05 16:20:00|
|U3     |kiran@mail.com|           |19 |Premium     |NULL  |2024-05-03 09:30:00|
+-------+--------------+-----------+---+------------+------+-------------------+



---
## Q8. Filter age between 18 and 30 (inclusive) AND subscription = 'Premium'

In [15]:
q8_result = df_users.filter(
    (F.col("age").between(18, 30)) & (F.col("subscription") == "Premium")
)
q8_result.show(truncate=False)

+-------+--------------+--------+---+------------+------+-------------------+
|user_id|email         |username|age|subscription|status|raw_timestamp      |
+-------+--------------+--------+---+------------+------+-------------------+
|U1     |asha@mail.com |asha    |28 |Premium     |active|2024-05-01 10:15:00|
|U3     |kiran@mail.com|        |19 |Premium     |NULL  |2024-05-03 09:30:00|
|U5     |sam@mail.com  |sam     |22 |Premium     |active|2024-05-05 16:20:00|
|U1     |asha@mail.com |asha    |28 |Premium     |active|2024-05-01 10:15:00|
+-------+--------------+--------+---+------------+------+-------------------+



`.between(18, 30)` is inclusive on both ends, so ages exactly 18 and exactly 30 both qualify — this correctly includes U3 (age 19) and U1 (age 28).

---
## Q9. Why handle nulls before `sum()` / `avg()`

**Answer:**

1. **Silent exclusion bias** — `sum()` and `avg()` skip nulls by default instead of erroring.
   `avg()` specifically divides by the count of non-null values, not the total row count, so
   unhandled nulls quietly shift both numerator and denominator.
2. **Incorrect downstream totals** — if a null logically means 0 ("no revenue recorded") but is
   left null, `sum()` happens to still be correct, but `avg()` is not: it averages only over
   non-null rows, inflating the result. Filling with 0 first makes `avg()` reflect the intended
   business definition.
3. **Fragmented groupBy results** — null keys in a groupBy column form their own null bucket,
   which can split or hide data quality issues if not addressed deliberately beforehand.
4. **Reproducibility** — choosing the null strategy (drop vs fill vs flag) before aggregating
   makes results auditable and consistent across runs, rather than relying on Spark's default
   null-skipping behavior.

This is exactly what produced the `NULL` average for Clothing in Q4 — the right fix is to decide
up front (e.g., `na.fill(0)` on `sale_amount`) before aggregating, as demonstrated in the Q15
pipeline below.

In [16]:
# Demonstration: same data, average WITH vs WITHOUT filling nulls first
print("Without filling nulls first (avg() silently skips the null row):")
df_sales.filter(F.col("product_category") == "Clothing").agg(F.avg("sale_amount")).show()

print("After filling nulls with 0 first (avg() now reflects intended business meaning):")
df_sales.na.fill({"sale_amount": 0}).filter(F.col("product_category") == "Clothing").agg(F.avg("sale_amount")).show()

Without filling nulls first (avg() silently skips the null row):
+-----------------+
| avg(sale_amount)|
+-----------------+
|2637.643682634729|
+-----------------+

After filling nulls with 0 first (avg() now reflects intended business meaning):
+-----------------+
| avg(sale_amount)|
+-----------------+
|2517.065685714284|
+-----------------+



---
## Q10. Cast `raw_timestamp` to `TimestampType` and rename to `event_time`

In [17]:
print("Schema before:")
df_users.printSchema()

df_users_ts = (
    df_users
    .withColumn("raw_timestamp", F.col("raw_timestamp").cast(TimestampType()))
    .withColumnRenamed("raw_timestamp", "event_time")
)

print("Schema after cast + rename:")
df_users_ts.printSchema()
df_users_ts.select("user_id", "event_time").show(truncate=False)

Schema before:
root
 |-- user_id: string (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)
 |-- age: long (nullable = true)
 |-- subscription: string (nullable = true)
 |-- status: string (nullable = true)
 |-- raw_timestamp: string (nullable = true)

Schema after cast + rename:
root
 |-- user_id: string (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)
 |-- age: long (nullable = true)
 |-- subscription: string (nullable = true)
 |-- status: string (nullable = true)
 |-- event_time: timestamp (nullable = true)

+-------+-------------------+
|user_id|event_time         |
+-------+-------------------+
|U1     |2024-05-01 10:15:00|
|U2     |2024-05-02 11:00:00|
|U3     |2024-05-03 09:30:00|
|U4     |2024-05-04 14:45:00|
|U5     |2024-05-05 16:20:00|
|U1     |2024-05-01 10:15:00|
+-------+-------------------+



---
## Q11. The Shuffle process and why groupBy is a wide transformation

**Answer:**

**Shuffle:** when `groupBy(...).agg(...)` (or `join`, `distinct`, `repartition`, `orderBy`, etc.)
runs, Spark must bring together all rows sharing the same key so they can be combined — but rows
for any given key are normally scattered across many partitions, possibly on different executor
nodes. The shuffle is the mechanism that performs this redistribution:

1. Each partition computes a partial result and tags records with a target partition, determined
   by a hash or range partitioner on the key.
2. Tagged records are written to local shuffle files on disk (**shuffle write**).
3. Each downstream task fetches the records meant for it from every other partition/executor
   across the network (**shuffle read**).
4. The receiving task combines all values for its key(s) into the final aggregated result.

This involves disk I/O, network transfer, and serialization — all expensive relative to a
transformation that only touches data already local to its partition.

**Narrow vs. wide transformations:**

- **Narrow** (`map`, `filter`, `withColumn`): each output partition depends on exactly one input
  partition. No cross-partition data movement, so Spark executes it within a single stage with
  no shuffle.
- **Wide** (`groupBy`, `join`, `distinct`, `repartition`, `orderBy`): each output partition can
  depend on many (potentially all) input partitions, because matching keys could be anywhere.
  This forces a stage boundary — data must be shuffled before the next stage can start.

`groupBy` is classified as a wide transformation precisely because it requires this cross-partition
redistribution of data, unlike a narrow transformation that can be computed entirely within each
partition independently.

In [18]:
# You can see the stage boundary caused by the shuffle in the physical plan:
df_sales.groupBy("product_category").agg(F.avg("sale_amount")).explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[product_category#79], functions=[avg(sale_amount#80)])
   +- Exchange hashpartitioning(product_category#79, 200), ENSURE_REQUIREMENTS, [plan_id=937]
      +- HashAggregate(keys=[product_category#79], functions=[partial_avg(sale_amount#80)])
         +- FileScan csv [product_category#79,sale_amount#80] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/mnt/acer/Shared/Devs/CEI_2026/Week-05-Data-Cleaning/sales_data.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<product_category:string,sale_amount:double>




---
## Q12. Remove rows with null `email` OR empty-string `username`

In [19]:
print("Before:")
df_users.select("user_id", "email", "username").show()

df_users_clean = df_users.filter(
    F.col("email").isNotNull() & (F.trim(F.col("username")) != "")
)

print("After removing rows with null email OR empty username:")
df_users_clean.select("user_id", "email", "username").show()

Before:
+-------+--------------+--------+
|user_id|         email|username|
+-------+--------------+--------+
|     U1| asha@mail.com|    asha|
|     U2|          NULL|   rohit|
|     U3|kiran@mail.com|        |
|     U4| neha@mail.com|    neha|
|     U5|  sam@mail.com|     sam|
|     U1| asha@mail.com|    asha|
+-------+--------------+--------+

After removing rows with null email OR empty username:
+-------+-------------+--------+
|user_id|        email|username|
+-------+-------------+--------+
|     U1|asha@mail.com|    asha|
|     U4|neha@mail.com|    neha|
|     U5| sam@mail.com|     sam|
|     U1|asha@mail.com|    asha|
+-------+-------------+--------+



`F.trim(...)` guards against whitespace-only usernames (`" "`) being missed by a plain `!= ""`
check. `isNotNull()` and the username condition are combined with `&` since **both** must
independently hold for a row to be kept — by De Morgan's law, `NOT(A OR B) = NOT A AND NOT B`,
which is exactly the logic implemented here.

---
## Q13. Using `.agg()` for multiple statistics at once (min, max, mean of `price`)

In [20]:
price_stats = df_store.agg(
    F.min("price").alias("min_price"),
    F.max("price").alias("max_price"),
    F.mean("price").alias("mean_price"),   # mean() is an alias for avg()
)
price_stats.show()

+---------+---------+----------+
|min_price|max_price|mean_price|
+---------+---------+----------+
|     50.0|    200.0|      95.0|
+---------+---------+----------+



`.agg()` accepts any number of aggregate column expressions in a single call, computing all of them in one pass over the data — avoiding three separate, more expensive passes that calling `.select(F.min(...))`, `.select(F.max(...))`, `.select(F.mean(...))` separately would require.

---
## Q14. Risk of `inferSchema=true` with messy/inconsistent date formats

**Answer:**

1. **Silent fallback to StringType** — if sampled rows mix formats (`"2024-01-05"`,
   `"05/01/2024"`, `"Jan 5 2024"`, or blanks), Spark usually cannot confidently infer
   `DateType`/`TimestampType` and falls back to plain string. Date-specific operations
   (`date_add`, `year()`, date range filters) then silently fail to apply correctly, with no
   error raised.
2. **Sampling bias** — schema inference scans only a sample of the data, so a type that fits the
   sample may break later once a stray bad value elsewhere in the full dataset is processed.
3. **Extra full read pass** — inferring schema requires reading the data once just to determine
   types, then again to load it — doubling I/O cost on large files, undermining some of the
   performance benefit Spark is meant to provide.
4. **Run-to-run inconsistency** — if the source file's format drifts slightly between runs, the
   same pipeline can infer a different schema each time, causing hard-to-debug downstream type
   mismatches.
5. **Numeric/date ambiguity** — a column like `"20240105"` could be inferred as `IntegerType`
   instead of representing a date, silently corrupting any date-based logic downstream.

**Best practice:** define an explicit `StructType` schema up front, or read the column as
`StringType` and explicitly parse it with `to_date(col, "fmt")` / `to_timestamp(...)`, rather than
relying on `inferSchema=true` for production pipelines with messy date data.

In [21]:
# Demonstration: explicit schema avoids the guesswork entirely
from pyspark.sql.types import StructType, StructField, StringType

explicit_schema = StructType([
    StructField("user_id", StringType(), True),
    StructField("raw_date_str", StringType(), True),  # kept as string deliberately
])

mixed_dates = [("U1", "2024-01-05"), ("U2", "05/01/2024"), ("U3", "Jan 5 2024")]
df_mixed = spark.createDataFrame(mixed_dates, schema=explicit_schema)
df_mixed.printSchema()
df_mixed.show()

# Parse explicitly per known format, using try_to_date so a mismatched format
# returns NULL instead of throwing -- this is the safe alternative to inferSchema's guesswork
df_mixed_parsed = df_mixed.withColumn(
    "parsed_date", F.coalesce(
        F.expr("try_to_date(raw_date_str, 'yyyy-MM-dd')"),
        F.expr("try_to_date(raw_date_str, 'dd/MM/yyyy')"),
        F.expr("try_to_date(raw_date_str, 'MMM d yyyy')"),
    )
)
df_mixed_parsed.show()

root
 |-- user_id: string (nullable = true)
 |-- raw_date_str: string (nullable = true)

+-------+------------+
|user_id|raw_date_str|
+-------+------------+
|     U1|  2024-01-05|
|     U2|  05/01/2024|
|     U3|  Jan 5 2024|
+-------+------------+

+-------+------------+-----------+
|user_id|raw_date_str|parsed_date|
+-------+------------+-----------+
|     U1|  2024-01-05| 2024-01-05|
|     U2|  05/01/2024| 2024-01-05|
|     U3|  Jan 5 2024| 2024-01-05|
+-------+------------+-----------+



---
## Q15. Final pipeline: remove duplicates → fill null prices with 0 → groupBy `store_id` → total revenue

In [22]:
print("Input df_store:")
df_store.show()

final_pipeline_result = (
    df_store
    .dropDuplicates()                            # Step 1: remove duplicate rows
    .na.fill({"price": 0})                       # Step 2: fill null prices with 0
    .groupBy("store_id")                         # Step 3: group by store_id
    .agg(F.sum("price").alias("total_revenue"))  # Step 3 cont.: total revenue
    .orderBy("store_id")
)

print("Final pipeline result:")
final_pipeline_result.show()

Input df_store:
+--------+-----+
|store_id|price|
+--------+-----+
|      S1|100.0|
|      S1|200.0|
|      S1| NULL|
|      S2| 50.0|
|      S2| 75.0|
|      S2| 50.0|
|      S3| NULL|
|      S3| NULL|
+--------+-----+

Final pipeline result:
+--------+-------------+
|store_id|total_revenue|
+--------+-------------+
|      S1|        300.0|
|      S2|        125.0|
|      S3|          0.0|
+--------+-------------+



**Verification:** after `dropDuplicates()`, S2's duplicate `50.0` row is removed (S2 keeps
`50.0 + 75.0 = 125.0`, not `175.0`). S3 has both prices null, which `na.fill(0)` converts to `0`
before summing, giving a defined `0.0` total revenue instead of `NULL` — this is the Q9 principle
applied directly in a real pipeline.

---
## Key Insights

- **In-memory caching is Spark's core advantage over MapReduce** — it converts repeated disk
  read/write cycles (the main bottleneck for iterative algorithms) into RAM-speed access after
  the first pass.
- **DataFrame immutability is not a limitation but an enabler** — it's what makes safe method
  chaining, lazy evaluation, and lineage-based fault recovery possible at the same time.
- **Null-handling strategy materially changes aggregation results** — the same dataset produced a
  `NULL` average (Q4) until nulls were deliberately filled with 0 in the final pipeline (Q15),
  which is why null handling must happen *before*, not after, calling `sum()`/`avg()`.
- **`groupBy`, `join`, and similar wide transformations are inherently more expensive** than narrow
  transformations like `filter` or `withColumn` because they require a shuffle — physically
  redistributing data across the cluster — so minimizing unnecessary wide transformations
  (e.g., filtering *before* grouping, not after) is a key performance practice.
- **Schema correctness and explicit cleaning order** (dedupe → null-handle → aggregate, as in
  Q15) together prevent silent data-quality bugs that wouldn't raise errors but would quietly
  produce wrong numbers.

In [23]:
spark.stop()